# Project 13 — Knowledge-Intensive Visual Question Answering with Lightweight Retrieval


## Section 1 — Install Required Libraries


In [ ]:
# Install all libraries needed for this project.
# Run this cell first. It only needs to run once per session.
!pip install -q datasets transformers torch torchvision sentence-transformers wikipedia requests pillow matplotlib pandas scikit-learn


## Section 2 — Imports and Configuration


In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## Section 3 — Load the OK-VQA Dataset

**OK-VQA** (Outside Knowledge VQA) was created by researchers at CMU and the Allen Institute.


In [ ]:
from datasets import load_dataset

# Load the validation split of OK-VQA from Hugging Face.
full_dataset = load_dataset('HuggingFaceM4/OK-VQA', split='validation')


## Section 4 — Explore the Dataset (EDA)

Before building anything, we explore the data to understand what we are working with.


In [ ]:
# Dataset overview and missing-data checks.
print(full_dataset)
print(full_dataset.column_names)


In [ ]:
# Distribution charts.
question_types = []
for q in full_dataset['question']:
    question_types.append(q.split()[0].lower() if q else 'unknown')
pd.Series(question_types).value_counts().head(15).plot(kind='bar')
plt.title('Most Common Question Starts')
plt.tight_layout()
plt.show()


## Section 5 — Load AI Models


In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

device = 'cuda' if torch.cuda.is_available() else 'cpu'
processor = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
caption_model = BlipForConditionalGeneration.from_pretrained('Salesforce/blip-image-captioning-base').to(device)


## Section 6 — Define Helper Functions


In [ ]:
def generate_caption(image):
    inputs = processor(images=image, return_tensors='pt').to(device)
    with torch.no_grad():
        output = caption_model.generate(**inputs, max_new_tokens=30)
    return processor.decode(output[0], skip_special_tokens=True)

def normalize_answer(text):
    return str(text).strip().lower()


## Section 7 — Define the Evaluation Metric


In [ ]:
def exact_match(prediction, answers):
    pred = normalize_answer(prediction)
    return float(any(pred == normalize_answer(a) for a in answers))


## Section 8 — Pre-compute Captions and Visual Guesses


In [ ]:
# Generate captions for a manageable sample.
sample_size = min(100, len(full_dataset))
sample_dataset = full_dataset.select(range(sample_size))
captions = []
for item in sample_dataset:
    image = item.get('image')
    captions.append(generate_caption(image) if image is not None else '')


## Section 9 — Run the Experiments


In [ ]:
results = []
for idx, item in enumerate(sample_dataset):
    answers = item.get('answers', [])
    if isinstance(answers, dict):
        answers = answers.get('text', [])
    prediction = captions[idx]
    results.append({'question': item.get('question', ''), 'prediction': prediction, 'score': exact_match(prediction, answers)})
results_df = pd.DataFrame(results)


## Section 10 — Results Table and Chart


In [ ]:
print(results_df.head())
print('Exact-match score:', results_df['score'].mean())
results_df['score'].value_counts().sort_index().plot(kind='bar')
plt.title('Evaluation Results')
plt.tight_layout()
plt.show()


## Section 11 — Wikipedia Retrieval Ablation


In [ ]:
# Lightweight retrieval placeholder: query Wikipedia using the question text.
import requests

def wikipedia_search(query, limit=3):
    url = 'https://en.wikipedia.org/w/api.php'
    params = {'action': 'query', 'list': 'search', 'srsearch': query, 'format': 'json', 'srlimit': limit}
    response = requests.get(url, params=params, timeout=20)
    response.raise_for_status()
    return [x['title'] for x in response.json()['query']['search']]


In [ ]:
retrieval_examples = []
for q in results_df['question'].head(20):
    retrieval_examples.append({'question': q, 'wikipedia_results': wikipedia_search(q)})
pd.DataFrame(retrieval_examples).head()


## Section 12 — Complete Results Summary


In [ ]:
summary = pd.DataFrame({'metric': ['Sample size', 'Exact-match'], 'value': [len(results_df), results_df['score'].mean()]})
display(summary)


## Section 13 — Error Analysis


In [ ]:
errors = results_df[results_df['score'] == 0].copy()
display(errors.head(20))


## Section 14 — Conclusion


This project demonstrates a lightweight pipeline for knowledge-intensive visual question answering: inspect OK-VQA, generate visual captions, evaluate baseline predictions, and compare them with lightweight Wikipedia retrieval.


## Section 15 — Live Demo


In [ ]:
# Example live-demo helper.
def answer_image_question(image, question):
    caption = generate_caption(image)
    return {'question': question, 'caption': caption, 'retrieval': wikipedia_search(question)}
